In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
print("All libraries imported successfully")

All libraries imported successfully


In [2]:
# Set random seed for reproducibility
np.random.seed(42)

# Number of transactions to generate
n = 1000

# Build the DataFrame column by column
df = pd.DataFrame({
    # Random amounts - exponential dist mimics real spending
    "amount": np.random.exponential(scale=500, size=n),
    
    # Randomly credit or debit
    "type": np.random.choice(["credit", "debit"], size=n),
    
    # Country with realistic probabilities
    "country": np.random.choice(
        ["US", "UK", "IN", "NG", "RU"],
        size=n,
        p=[0.5, 0.2, 0.15, 0.1, 0.05]
    ),
    
    # Customer age between 18 and 70
    "customer_age": np.random.randint(18, 70, size=n),
    
    # Hour of transaction 0-23
    "hour": np.random.randint(0, 24, size=n)
})

# Start everyone as legitimate (0)
df["fraud"] = 0

# Flag high amount from high risk countries as fraud
df.loc[(df["amount"] > 1500) & 
       (df["country"].isin(["NG", "RU"])), "fraud"] = 1

# Flag high amount at unusual hours as fraud
df.loc[(df["amount"] > 2000) & 
       (df["hour"] < 6), "fraud"] = 1

# Check shape and fraud distribution
print(df.shape)
print(df["fraud"].value_counts())
print(df.head())

(1000, 6)
fraud
0    988
1     12
Name: count, dtype: int64
        amount    type country  customer_age  hour  fraud
0   234.634045  credit      UK            68    12      0
1  1505.060715   debit      US            64     7      0
2   658.372847   debit      US            64     6      0
3   456.471277   debit      US            21    16      0
4    84.812435   debit      US            24    20      0


#### Feature Preparation

In [3]:
# Step 1 - Make a copy so original data is preserved
df_model = df.copy()

# Step 2 - Encode text columns to numbers
le = LabelEncoder()

# credit/debit → 1/0
df_model["type"] = le.fit_transform(df_model["type"])

# US/UK/IN/NG/RU → 4/3/0/1/2 (alphabetical order)
df_model["country"] = le.fit_transform(df_model["country"])

# Step 3 - Separate features (X) and label (y)
# X = what the model uses to make predictions
# y = what we want the model to predict

X = df_model[["amount", "type", "country", "customer_age", "hour"]]
y = df_model["fraud"]

# Step 4 - Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
   X, y,
    test_size= 0.2,
    random_state= 42,
    stratify=y
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)
print("\nFraud cases in training:", y_train.sum())
print("Fraud cases in testing:", y_test.sum())

Training set size: (800, 5)
Testing set size: (200, 5)

Fraud cases in training: 10
Fraud cases in testing: 2


#### Train and Evaluate

In [4]:
# Step 1 - Create the model
model = LogisticRegression(random_state=42, max_iter=1000)

# Step 2 - Train the model
model.fit(X_train, y_train)

# Step 3 - Make predictions on test set
y_pred = model.predict(X_test)

# Step 4 - Evaluate performance
print("===confusion matrix===")
print(confusion_matrix(y_test, y_pred))

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

===confusion matrix===
[[196   2]
 [  1   1]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       198
           1       0.33      0.50      0.40         2

    accuracy                           0.98       200
   macro avg       0.66      0.74      0.70       200
weighted avg       0.99      0.98      0.99       200



Precision 0.33 for fraud:

Of all transactions flagged as fraud, only 33% were actually fraud. 2 flags raised, only 1 was real fraud.
Recall 0.50 for fraud:

Of all actual frauds, we caught 50%. Missed 1 out of 2 fraud cases.
Accuracy 0.98:

Looks impressive but misleading — model is mostly just predicting everything as legitimate.

##### Add Class Weights

In [5]:
# Tell model to penalize missing fraud more heavily
model_weighted = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced" # automatically handles imbalance
)

model_weighted.fit(X_train, y_train)
y_pred_weighted = model_weighted.predict(X_test)

print("===confusion matrix===")
print(confusion_matrix(y_test, y_pred_weighted))

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred_weighted))

===confusion matrix===
[[192   6]
 [  0   2]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       1.00      0.97      0.98       198
           1       0.25      1.00      0.40         2

    accuracy                           0.97       200
   macro avg       0.62      0.98      0.69       200
weighted avg       0.99      0.97      0.98       200



##### We can conclude that model 2 is better than model 1

#### Feature Engineering

In [6]:
# Work on a fresh copy
df_features = df.copy()

# Feature 1 - Night transaction (12am to 6am)
df_features["is_night"] = (df_features["hour"] < 6).astype(int)

# Feature 2 - High value transaction
df_features["is_high_amount"] = (df_features["amount"] > 1500).astype(int)

# Feature 3 - Amount z-score (how unusual is this amount)
df_features["amount_zscore"] = ((df_features["amount"] - df_features["amount"].mean()) /
                                df_features["amount"].std())

# Feature 4 - High risk country
df_features["is_high_risk_country"] = (df_features["country"].isin(["NG", "RU"])).astype(int)

# Feature 5 - High risk combination flag
df_features["is_high_risk"] = ((df_features["is_night"] == 1) & (df_features["is_high_amount"] == 1)).astype(int)

# Show new features
print(df_features[["amount", "hour", "is_night", 
                    "is_high_amount", "amount_zscore",
                    "is_high_risk_country", 
                    "is_high_risk", "fraud"]].head(10))

        amount  hour  is_night  is_high_amount  amount_zscore  \
0   234.634045    12         0               0      -0.517463   
1  1505.060715     7         0               1       2.095213   
2   658.372847     6         0               0       0.353970   
3   456.471277    16         0               0      -0.061247   
4    84.812435    20         0               0      -0.825576   
5    84.798146    22         0               0      -0.825606   
6    29.919384    14         0               0      -0.938466   
7  1005.615432    19         0               0       1.068087   
8   459.541077    14         0               0      -0.054934   
9   615.625031     0         1               0       0.266058   

   is_high_risk_country  is_high_risk  fraud  
0                     0             0      0  
1                     0             0      0  
2                     0             0      0  
3                     0             0      0  
4                     0             0      0  
5 

#### Retrain model with new features:

In [7]:
# Step 1 - Encode country and type columns
df_model2 = df_features.copy()
le = LabelEncoder()
df_model2["country"] = le.fit_transform(df_model2["country"])
df_model2["type"] = le.fit_transform(df_model2["type"])

# Step 2 - New feature set including engineered features
X_new = df_model2[["amount", "type", "country", 
                    "customer_age", "hour",
                    "is_night", "is_high_amount",
                    "amount_zscore", "is_high_risk_country",
                    "is_high_risk"]]
y_new = df_model2["fraud"]

# Step 3 - Split data
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_new, y_new,
    test_size=0.2,
    random_state=42,
    stratify=y_new
)

# Step 4 - Train improved model
model2 = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

model2.fit(X_train2,y_train2)

# Step 5 - Evaluate
y_pred2 = model2.predict(X_test2)

print("=== Model with Engineered Features ===")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test2, y_pred2))
print("\nClassification Report:")
print(classification_report(y_test2, y_pred2))

=== Model with Engineered Features ===

Confusion Matrix:
[[195   3]
 [  0   2]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       198
           1       0.40      1.00      0.57         2

    accuracy                           0.98       200
   macro avg       0.70      0.99      0.78       200
weighted avg       0.99      0.98      0.99       200



In [8]:
import joblib

# Save the best model
joblib.dump(model2, "fraud_model.pkl")
print("Model saved successfully")

Model saved successfully


In [9]:
# Save the label encoder too
joblib.dump(le, "label_encoder.pkl")
print("Label encoder saved successfully")

Label encoder saved successfully


#### Decision Tree

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_text

dt_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
    class_weight= "balanced"
)

#Train
dt_model = dt_model.fit(X_train2, y_train2)

#Predict
y_pred_dt = dt_model.predict(X_test2)

# Evaluate
print("=== Decision Tree Results ===")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test2, y_pred_dt))
print("\nClassification Report:")
print(classification_report(y_test2, y_pred_dt))

# Print the actual decision rules
print("\n=== Decision Tree Rules ===")
feature_names = X_new.columns.tolist()
print(export_text(dt_model, feature_names=feature_names))

=== Decision Tree Results ===

Confusion Matrix:
[[195   3]
 [  2   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       198
           1       0.00      0.00      0.00         2

    accuracy                           0.97       200
   macro avg       0.49      0.49      0.49       200
weighted avg       0.98      0.97      0.98       200


=== Decision Tree Rules ===
|--- is_high_amount <= 0.50
|   |--- class: 0
|--- is_high_amount >  0.50
|   |--- customer_age <= 36.50
|   |   |--- class: 0
|   |--- customer_age >  36.50
|   |   |--- country <= 0.50
|   |   |   |--- class: 0
|   |   |--- country >  0.50
|   |   |   |--- hour <= 22.50
|   |   |   |   |--- class: 1
|   |   |   |--- hour >  22.50
|   |   |   |   |--- class: 0



In [11]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest with 100 trees
rf_model = RandomForestClassifier(
    random_state=42,
    max_depth=5,
    n_estimators=100,
    class_weight="balanced"
)

# Train
rf_model.fit(X_train2,y_train2)

# Predict
y_pred_rf = rf_model.predict(X_test2)

# Evaluate
print("=== Random Forest Results ===")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test2, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test2, y_pred_rf))

=== Random Forest Results ===

Confusion Matrix:
[[197   1]
 [  1   1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       198
           1       0.50      0.50      0.50         2

    accuracy                           0.99       200
   macro avg       0.75      0.75      0.75       200
weighted avg       0.99      0.99      0.99       200



In [12]:
# Get feature importance scores

importances = pd.DataFrame({
    "feature": X_new.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending= False)

print(importances)

                feature  importance
7         amount_zscore    0.299075
6        is_high_amount    0.282299
0                amount    0.272028
8  is_high_risk_country    0.051475
9          is_high_risk    0.036715
2               country    0.026586
3          customer_age    0.021079
4                  hour    0.006409
1                  type    0.002410
5              is_night    0.001925


##### Cross Validation

In [13]:
from sklearn.model_selection import cross_val_score

# Run 5-fold cross validation on Random Forest

cv_scores = cross_val_score(
    rf_model, X_new, y_new,
    cv=5,
    scoring="f1"    
)

print("F1 scores across 5 folds:", cv_scores)
print("Average F1 score:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

F1 scores across 5 folds: [0.66666667 1.         1.         1.         0.8       ]
Average F1 score: 0.8933333333333333
Standard deviation: 0.13727506854649335
